In [28]:
from pyspark.sql import SparkSession

import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Administrator\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Administrator\AppData\Local\Programs\Python\Python310\python.exe"

spark = (
    SparkSession.builder
    .appName("pyspark-local")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark context Web UI available at:", sc.uiWebUrl)
print("Spark context available as 'sc'")
print("SparkSession available as 'spark'")

Spark context Web UI available at: http://host.docker.internal:4040
Spark context available as 'sc'
SparkSession available as 'spark'


In [6]:
nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
rdd = sc.parallelize(nums)
# 2. Basic actions
print("Count:", rdd.count()) # 10
print("First 3:", rdd.take(3)) # [1, 2, 3]
print("Sample:", rdd.takeSample(False, 3)) # Random 3

# Create RDD from the list
data = ["Spark", "RDD", "HandsOn"]
rdd_new = sc.parallelize(data)
# Print the count
print(rdd_new.count())

Count: 10
First 3: [1, 2, 3]
Sample: [10, 6, 2]
10


In [10]:
# MAP: Transform each element
squared = rdd.map(lambda x: x*x)
print("Squared:", squared.collect()) # [1, 4, 9, 16...]
# FILTER: Select subset
evens = rdd.filter(lambda x: x%2 == 0)
print("Evens:", evens.collect()) # [2, 4, 6, 8, 10]
# Chain them
big_squares = rdd.map(lambda x: x*x).filter(lambda x: x > 50)
print("Big squares:", big_squares.collect())

cubes = rdd.map(lambda x: x*x*x).filter(lambda x: x > 100)
print("Cubes:", cubes.collect()) # [1, 4, 9, 16...]
total = rdd.reduce(lambda a,b: a+b)
print("Sum:", total) # 55


Squared: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
Evens: [2, 4, 6, 8, 10]
Big squares: [64, 81, 100]
Cubes: [125, 216, 343, 512, 729, 1000]
Sum: 55


In [16]:
lines = [
"Apache Spark processes data fast",
"RDD is resilient distributed dataset",
"Hands-on practice makes perfect",
"Transformations are lazy evaluations"
]
text_rdd = sc.parallelize(lines)
print("Lines:", text_rdd.collect())

# FLATMAP: Split + flatten
words = text_rdd.flatMap(lambda line: line.lower().split())
print("All words:", words.collect())
print("First 5:", words.take(5))

#Clean words: lowercase + remove "data" + length > 3
clean_words = text_rdd.map(lambda word: word.lower()).filter(lambda word: word != "data").filter(lambda word: len(word) > 3)
print("Clean words:", clean_words.collect())

#Count words with length > 4
long_words_count = text_rdd.map(lambda word: word.split()).flatMap(lambda x: x).filter(lambda w: len(w) > 4).count()
print("Count of words with length > 4:", long_words_count)

Lines: ['Apache Spark processes data fast', 'RDD is resilient distributed dataset', 'Hands-on practice makes perfect', 'Transformations are lazy evaluations']
All words: ['apache', 'spark', 'processes', 'data', 'fast', 'rdd', 'is', 'resilient', 'distributed', 'dataset', 'hands-on', 'practice', 'makes', 'perfect', 'transformations', 'are', 'lazy', 'evaluations']
First 5: ['apache', 'spark', 'processes', 'data', 'fast']
Clean words: ['apache spark processes data fast', 'rdd is resilient distributed dataset', 'hands-on practice makes perfect', 'transformations are lazy evaluations']
Count of words with length > 4: 12


In [30]:
# 1. Create (word, 1) pairs
word_pairs = words.map(lambda w: (w, 1))
# 2. Count by word
counts = word_pairs.reduceByKey(lambda a,b: a+b)
print("Word counts:", counts.collect())
# 3. Sort by frequency (descending)
top_words = counts.map(lambda kv: (kv[1], kv[0])) \
 .sortByKey(ascending=False)
print("Top 3:", top_words.take(3))

# Filter words appearing 2+ times only
# counts = word_pairs.reduceByKey(lambda a, b: a + b)

# Filter words with frequency >= 2
frequent_words = counts.filter(lambda kv: kv[1] >= 2)

print("Frequent words (2+ times):")
print(frequent_words.collect())
#frequent_words.saveAsTextFile("frequent_wordcount_output")



Word counts: [('hands-on', 1), ('perfect', 1), ('are', 1), ('spark', 1), ('dataset', 1), ('practice', 1), ('makes', 1), ('lazy', 1), ('processes', 1), ('is', 1), ('transformations', 1), ('evaluations', 1), ('apache', 1), ('data', 1), ('fast', 1), ('rdd', 1), ('resilient', 1), ('distributed', 1)]
Top 3: [(1, 'hands-on'), (1, 'perfect'), (1, 'are')]
Frequent words (2+ times):
[]


In [32]:
transactions = [
 ("Alice", 100), ("Bob", 200), ("Alice", 50),
 ("Charlie", 70), ("Bob", 30), ("Alice", 150)
]
tx_rdd = sc.parallelize(transactions)

total_spend = tx_rdd.reduceByKey(lambda a, b: a + b)

print("Total spend per customer:")
print(total_spend.collect())


sorted_spend = total_spend \
    .map(lambda kv: (kv[1], kv[0])) \
    .sortByKey(ascending=False)

print("Sorted by spend (DESC):")
print(sorted_spend.collect())


top_2 = sorted_spend.take(2)

print("Top 2 customers:")
print(top_2)

Total spend per customer:
[('Bob', 230), ('Charlie', 70), ('Alice', 300)]
Sorted by spend (DESC):
[(300, 'Alice'), (230, 'Bob'), (70, 'Charlie')]
Top 2 customers:
[(300, 'Alice'), (230, 'Bob')]
